In [1]:
from os import getcwd
from os.path import join, dirname, exists
import pandas as pd
from scipy.stats import wasserstein_distance
import numpy as np
from scipy.spatial.distance import cdist
import ot
from sklearn.cluster import KMeans
from sklearn.neighbors import KernelDensity
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.dates import date2num, DateFormatter, MonthLocator
import json
import pickle
from collections import defaultdict
from itertools import product, cycle
from pprint import pprint
from math import ceil, sqrt
import textwrap
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import random
# from matplotlib import colormaps

In [17]:
# Define all relevant electricity use columns for the table
all_electricity_cols = [
    "End Use: Electricity: Heating", "End Use: Electricity: Heating Fans/Pumps",
    "End Use: Electricity: Heating Heat Pump Backup", "End Use: Electricity: Heating Heat Pump Backup Fans/Pumps",
    "End Use: Electricity: Cooling", "End Use: Electricity: Cooling Fans/Pumps",
    "End Use: Electricity: Hot Water", "End Use: Electricity: Lighting Interior",
    "End Use: Electricity: Lighting Exterior", "End Use: Electricity: Appliances",
    "End Use: Electricity: Plug Loads", "End Use: Electricity: Photovoltaics"
]

def find_peak_electricity_use(region_name, num_residences, residence_type="all", casename=None, offset_needed=False):
    """
    Searches through RESSTOCK building data to find the single building 
    with the highest total electricity consumption at a single hour.

    Args:
        region_name (str): The name of the city for the data.
        num_residences (int): The total number of residences in the dataset.
        residence_type (str, optional): The type of residence. Defaults to "all".
        casename (str, optional): The case name for the dataset. Defaults to None.
        offset_needed (bool, optional): Indicates if an offset is needed for file paths. Defaults to False.

    Returns:
        tuple: A tuple containing a string with a summary of the peak-use building 
               and a pandas DataFrame of the electricity use breakdown for that hour.
    """
    
    max_total_electricity = 0
    peak_hour_data = None
    peak_building_id = None
    peak_building_metadata = None

    
                
    # Retrieve additional metadata from the centralized buildstock CSV file
    resstock_buildstock_data_path = join("resources", "national", "all_national_8000_all_unit_buildstock.csv")
    resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)
    
    residences_to_collect = range(num_residences)

    for i in residences_to_collect:
        if offset_needed:
            j = i + 1 + num_residences
        else:
            j = i + 1
        
        filepath = join(f"{region_name}_{num_residences}_{residence_type}_unit_residence{'' if casename is None else f'_{casename}'}", f'run{j}','run', 'results_timeseries.csv')
        
        try:
            timeseries_data = pd.read_csv(filepath, low_memory=True, header=[0,1])
        except Exception as e:
            continue

        # Clean up column names by dropping the second level
        timeseries_data.columns = timeseries_data.columns.droplevel(1)

        # Check for the existence of the total electricity column
        if "Fuel Use: Electricity: Total" in timeseries_data.columns:
            # Find the index of the maximum total electricity consumption
            max_index = timeseries_data["Fuel Use: Electricity: Total"].idxmax()
            current_max_use = timeseries_data.loc[max_index]["Fuel Use: Electricity: Total"]
            
            # Compare and update if a new peak is found
            if current_max_use > max_total_electricity:
                max_total_electricity = current_max_use
                peak_hour_data = timeseries_data.loc[max_index]
                
                # Get the building ID from the JSON metadata file
                data_point_out_filepath = join(f"{region_name}_{num_residences}_{residence_type}_unit_residence{'' if casename is None else f'_{casename}'}", f'run{j}','run', 'data_point_out.json')
                try:
                    with open(data_point_out_filepath, "r") as f:
                        json_data = json.load(f)
                        peak_building_id = json_data["BuildExistingModel"]["building_id"]
                except Exception as e:
                    peak_building_id = "N/A"
                peak_building_metadata = resstock_buildstock_data[resstock_buildstock_data["Building"] == peak_building_id]
                
    # Prepare the output
    if peak_hour_data is not None:        
        return peak_hour_data, peak_building_metadata
    else:
        return "No data found to process.", None

In [21]:
peak_hour_data, peak_building_metadata  = find_peak_electricity_use("national_ASHP", 8000, casename=None)

/tmp/ipykernel_2178/1957883278.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [22]:
# Define all relevant electricity use columns for the table
all_electricity_cols = [
    "End Use: Electricity: Heating", "End Use: Electricity: Heating Fans/Pumps",
    "End Use: Electricity: Heating Heat Pump Backup", "End Use: Electricity: Heating Heat Pump Backup Fans/Pumps",
    "End Use: Electricity: Cooling", "End Use: Electricity: Cooling Fans/Pumps",
    "End Use: Electricity: Hot Water", "End Use: Electricity: Lighting Interior",
    "End Use: Electricity: Lighting Exterior", "End Use: Electricity: Appliances",
    "End Use: Electricity: Plug Loads", "End Use: Electricity: Photovoltaics"
]

# Create the breakdown table for the peak hour
breakdown_data = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data.index],
    "Electricity (kWh)": [peak_hour_data[col] for col in all_electricity_cols if col in peak_hour_data.index]
}
breakdown_table = pd.DataFrame(breakdown_data)

breakdown_table

# # Generate a summary string
# peak_time = pd.to_datetime(peak_hour_data["Time"])
# summary = f"🥇 The building with the highest peak electricity consumption is building ID: {peak_building_metadata['Building']}\n"
# summary += f"Location: {peak_building_metadata['city'].iloc[0]}, {peak_building_metadata['state'].iloc[0]}\n"
# summary += f"The peak consumption was {peak_hour_data['Fuel Use: Electricity: Total']:.2f} kWh at {peak_time.strftime('%Y-%m-%d %H:%M')}.\n\n"
# summary += f"Breakdown of Electricity Use during Peak Hour:\n"

# summary

,End Use,Electricity (kWh)
0,Heating,18.751
1,Heating Fans/Pumps,1.858
2,Heating Heat Pump Backup,81.240
3,Heating Heat Pump Backup Fans/Pumps,1.964
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Hot Water,0.000


In [23]:
peak_hour_data_ASHP, peak_building_metadata_ASHP  = find_peak_electricity_use("national_ASHP", 8000, casename="ASHP")

/tmp/ipykernel_2178/1957883278.py:37: DtypeWarning: Columns (76,77,78,132) have mixed types. Specify dtype option on import or set low_memory=False.
  resstock_buildstock_data = pd.read_csv(resstock_buildstock_data_path)


In [24]:
# Create the breakdown table for the peak hour
breakdown_data_ASHP = {
    "End Use": [col.replace("End Use: Electricity: ", "") for col in all_electricity_cols if col in peak_hour_data_ASHP.index],
    "Electricity (kWh)": [peak_hour_data_ASHP[col] for col in all_electricity_cols if col in peak_hour_data_ASHP.index]
}
breakdown_table_ASHP = pd.DataFrame(breakdown_data_ASHP)

breakdown_table_ASHP

,End Use,Electricity (kWh)
0,Heating,19.106
1,Heating Fans/Pumps,1.845
2,Heating Heat Pump Backup,82.193
3,Heating Heat Pump Backup Fans/Pumps,1.978
4,Cooling,0.000
5,Cooling Fans/Pumps,0.000
6,Hot Water,0.000


In [25]:
peak_building_metadata

,Building,ASHRAE IECC Climate Zone 2004,Bathroom Spot Vent Hour,County and PUMA,AIANNH Area,CEC Climate Zone,City,County,AHS Region,ASHRAE IECC Climate Zone 2004 - 2A Split,...,Misc Pool Pump,Refrigerator,Roof Material,State Metro Median Income,Water Heater Fuel,Water Heater Efficiency,Water Heater In Unit,Water Heater Location,Window Areas,Windows
3494,3495,3A,Hour19,"G3701250, G37003700",No,NaN,In another census Place,"NC, Moore County",Non-CBSA South Atlantic,3A,...,NaN,EF 19.9,"Asphalt Shingles, Medium",Not Available,Electricity,Electric Standard,Yes,Living Space,F12 B12 L12 R12,"Single, Clear, Non-metal"


In [26]:
peak_building_metadata_ASHP

,Building,ASHRAE IECC Climate Zone 2004,Bathroom Spot Vent Hour,County and PUMA,AIANNH Area,CEC Climate Zone,City,County,AHS Region,ASHRAE IECC Climate Zone 2004 - 2A Split,...,Misc Pool Pump,Refrigerator,Roof Material,State Metro Median Income,Water Heater Fuel,Water Heater Efficiency,Water Heater In Unit,Water Heater Location,Window Areas,Windows
3494,3495,3A,Hour19,"G3701250, G37003700",No,NaN,In another census Place,"NC, Moore County",Non-CBSA South Atlantic,3A,...,NaN,EF 19.9,"Asphalt Shingles, Medium",Not Available,Electricity,Electric Standard,Yes,Living Space,F12 B12 L12 R12,"Single, Clear, Non-metal"


In [27]:
peak_hour_data_ASHP

Time                                                                                    2022-02-04T21:00:00
TimeDST                                                                                 2022-02-04T21:00:00
TimeUTC                                                                                2022-02-05T02:00:00Z
Energy Use: Total                                                                                    358.79
Energy Use: Net                                                                                      358.79
Fuel Use: Electricity: Total                                                                        105.151
Fuel Use: Electricity: Net                                                                          105.151
End Use: Electricity: Heating                                                                        19.106
End Use: Electricity: Heating Fans/Pumps                                                              1.845
End Use: Electricity: Heatin

In [31]:
peak_building_metadata_ASHP['Geometry Floor Area']

3494    4000+
Name: Geometry Floor Area, dtype: object